# 2. Production Reality — Optimized / Streaming Pipeline

**Agenda slot: 20–35 min**

Most production voice AI companies today still don't use end-to-end speech
models. They use separate STT + LLM + TTS — it gives better control, lower
cost, and more flexibility. The "it's slow" story from notebook 1 is only
true if you implement it naively (wait for each stage to fully finish).
Modern systems optimize every stage instead.

Full agenda: [`../voice-agent.md`](../voice-agent.md)

## What a production-grade modular pipeline looks like

```
User speaks
 ↓
Streaming STT (Deepgram, Gladia, Speechmatics, Google, AssemblyAI)
 ↓
Partial transcripts every 100-300ms
 ↓
LLM starts reasoning before user finishes speaking
 ↓
Tool calls execute in parallel
 ↓
Streaming TTS (ElevenLabs, Cartesia, OpenAI TTS, Azure)
 ↓
Audio streamed back immediately
```

### The five key optimizations

**1. Streaming STT** — instead of waiting for the user to stop talking, the
STT service continuously emits partial transcripts:

```
200ms → "I want"
400ms → "I want to"
700ms → "I want to book"
1.0s  → "I want to book a"
1.2s  → "I want to book a flight"
```

The LLM can begin processing before the sentence is complete. This needs a
dedicated streaming STT provider (Deepgram, AssemblyAI, etc.) — OpenAI's
Whisper API used in notebook 1 is request/response only, so we can't demo
this part live without another provider's key. The two optimizations below,
though, we *can* demo directly with the OpenAI SDK already in this project.

**2. Incremental LLM generation** — demoed below.

**3. Streaming TTS** — demoed below.

**4. Parallel tool execution** — if the user asks "What's the weather in
Pune?", a well-designed system identifies the tool call quickly, starts the
API request, and prepares the response template while the call is in flight —
instead of waiting for a long response before even starting the lookup.

**5. Voice Activity Detection (VAD)** — determines when the user has likely
finished speaking, rather than waiting for a long silence. Shaves hundreds of
milliseconds off every turn.

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import time

load_dotenv()
client = OpenAI()

question = "What's the weather like in Pune today, and what should I wear?"

### Demo: incremental LLM generation

Watch tokens print as they arrive, instead of waiting for the full response.

In [ ]:
t0 = time.time()
stream = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": question}], stream=True)

first_token_at = None
answer = ""
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        first_token_at = first_token_at or time.time()
        answer += delta
        print(delta, end="", flush=True)

print(f"\n\nFirst token after {first_token_at - t0:.2f}s")

### Demo: streaming TTS

Audio bytes start arriving before the full response is synthesized. In a real
app you'd play chunks as they land (that's the overlap that makes production
systems feel instant); here we time it and buffer to a file for simplicity.

In [ ]:
import sounddevice as sd
import soundfile as sf

t0 = time.time()
with client.audio.speech.with_streaming_response.create(model="tts-1", voice="alloy", input=answer, response_format="wav") as response:
    response.stream_to_file("stream_answer.wav")
print(f"Streaming TTS finished in {time.time() - t0:.2f}s")

audio, samplerate = sf.read("stream_answer.wav")
sd.play(audio, samplerate)
sd.wait()

## Typical latency breakdown

A well-optimized modular pipeline might look like:

| Stage | Latency |
|---|---|
| Streaming STT | 150–300 ms |
| LLM first token | 150–400 ms |
| Streaming TTS first audio | 100–200 ms |

Users often hear the first spoken response in **400–900 ms** overall — a far
cry from notebook 1's multi-second wait.

## Then why use GPT Realtime at all?

Its value isn't just lower latency — it's **simplifying the architecture**.

With a modular stack, you manage: mic → STT provider → LLM → TTS provider →
audio player. That means handling different APIs, synchronization, partial
transcripts, audio formats, barge-in, conversation state, and timing between
every component yourself.

With GPT Realtime: mic → GPT Realtime → speaker. One session handles speech
recognition, language understanding, response generation, speech synthesis,
streaming, interruptions, turn detection, and conversation management. The
trade-off is **simplicity versus flexibility**.

## Why many companies still choose separate STT + LLM + TTS

Because they can optimize each component independently:

- Use the best STT for noisy call centers
- Use a domain-specific LLM for reasoning
- Choose a TTS voice that matches their brand
- Swap providers without redesigning the whole system
- Reduce costs by selecting cheaper services per stage

Common production stacks:

```
Deepgram → Claude / GPT / Llama → Cartesia
Gladia   → Gemini               → ElevenLabs
```

### Two valid architectures

**Architecture 1 — Modular Voice Stack** (most common in production)

| Pros | Cons |
|---|---|
| Lower operational cost in many cases | More engineering effort |
| Best-in-class components | Multiple APIs and failure points |
| Easier to customize | More orchestration required |
| Provider flexibility | |

**Architecture 2 — Unified Realtime Model**

| Pros | Cons |
|---|---|
| Much simpler architecture | Less control over individual components |
| Built-in interruption handling | Tied to a single provider's capabilities |
| Native multimodal interaction | May not be the most cost-effective at very high scale |
| Faster to prototype and launch | |

Both are legitimate choices in production today — which one wins depends on
scale, budget, and how much control you need over each stage.

**Next up:** `03_gpt_realtime.ipynb` — we build with the Unified Realtime
Model, and switch to the actual browser app for the full end-to-end demo.